# Stage 02 · Step 3 — Surrogate-driven τ search

Use the SSL+RUL model to predict component-level remaining life on observed telemetry, derive an expected number of preventive / corrective events for any candidate τ schedule, and let Optuna optimise τ over that surrogate without paying the simulator's full cost.

The surrogate's predictions are validated against the **real simulator** at the end: the top-K τ vectors found by the surrogate are re-simulated with `lib.env_runner.run_with_tau` and scored with `lib.objective.scalar_objective`, then compared against Stage 01's leaderboard.

If the surrogate is well-calibrated, Stage 02 returns a τ vector whose true cost is within a few percent of Stage 01's optimum — at a fraction of the wall-clock time spent inside the optimiser.

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from transformers import PatchTSTConfig, PatchTSTForRegression

from ml_models.lib.data import (
    DEFAULT_FLEET_PATH,
    TEST_PRINTERS,
    VAL_PRINTERS,
    filter_printers,
    load_fleet,
)
from ml_models.lib.env_runner import default_dates, run_with_tau
from ml_models.lib.features import build_feature_matrix
from ml_models.lib.objective import scalar_objective
from ml_models import PROJECT_ROOT
from sdg.generate import build_printer_city_map, load_configs
from sdg.schema import COMPONENT_IDS

MODELS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/models'
RESULTS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
with (MODELS_DIR / 'ssl_config.json').open() as handle:
    saved = json.load(handle)
patch_cfg = PatchTSTConfig(**saved['patch_cfg'])
patch_cfg.num_targets = 6
patch_cfg.prediction_length = 1
patch_cfg.use_cls_token = False
model = PatchTSTForRegression(patch_cfg).to(device)
model.load_state_dict(torch.load(MODELS_DIR / 'rul_head_ssl.pt', map_location=device))
model.eval()

scaler = np.load(MODELS_DIR / 'feature_scaler.npz', allow_pickle=True)
channel_mean = scaler['mean']; channel_std = scaler['std']
CONTEXT_LENGTH = int(saved['train_cfg']['context_length'])
print('Loaded RUL model. Context length:', CONTEXT_LENGTH)

In [ ]:
fleet = load_fleet(DEFAULT_FLEET_PATH)
feat_fleet, feature_cols = build_feature_matrix(fleet)

def predict_rul_for_window(printer_id: int, end_day: int) -> np.ndarray:
    df = filter_printers(feat_fleet, [printer_id])
    df = df.sort_values('day')
    arr = df[feature_cols].to_numpy(dtype=np.float32)
    if end_day < CONTEXT_LENGTH:
        return np.full(6, np.nan, dtype=np.float32)
    window = arr[end_day - CONTEXT_LENGTH:end_day]
    normed = (window - channel_mean) / channel_std
    x = torch.from_numpy(normed).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(past_values=x).regression_outputs.squeeze(-1).view(-1)
    return (out.cpu().numpy() * 365.0).clip(0.0, 365.0)

sample_rul = predict_rul_for_window(VAL_PRINTERS[0], end_day=2000)
print('Predicted RUL for printer', VAL_PRINTERS[0], 'at day 2000:', sample_rul)

## Surrogate cost model

For each candidate τ vector and each held-out printer, estimate annual preventive and corrective events using:

$$n_{prev,i} \approx H \cdot (24 / \tau_i) \qquad n_{corr,i} \approx H \cdot \hat{\lambda}_{cm,i}(\tau_i)$$

where $H$ is the simulated horizon (years), and $\hat{\lambda}_{cm,i}$ is a smooth function of $\tau_i$ fit from the RUL predictions: rapidly decreasing for short τ (frequent maintenance prevents failures), increasing for long τ (more failures slip through). The simplest such fit uses a single observation: at the baseline τ, observed corrective events are known from `fleet_baseline.parquet`; at τ→∞, every component eventually fails and contributes a corrective event per nominal life. We interpolate between these regimes via the predicted RUL on validation printers.

In [ ]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
components = components_cfg['components']
DATES = default_dates()
HORIZON_YEARS = len(DATES) / 365.25

anchor_pm: dict[str, int] = {}
anchor_cm: dict[str, int] = {}
for component_id in COMPONENT_IDS:
    anchor_pm[component_id] = int(filter_printers(fleet, VAL_PRINTERS)[f'maint_{component_id}'].sum())
    anchor_cm[component_id] = int(filter_printers(fleet, VAL_PRINTERS)[f'failure_{component_id}'].sum())
anchor_tau = {component_id: float(components[component_id]['tau_nom_h']) for component_id in COMPONENT_IDS}
anchor_pm, anchor_cm, anchor_tau

In [ ]:
def surrogate_score(tau_vector: dict[str, float]) -> dict:
    n_printers = len(VAL_PRINTERS)
    pm_total = 0.0
    cm_total = 0.0
    downtime_total = 0.0
    for component_id in COMPONENT_IDS:
        spec = components[component_id]
        tau_new = float(tau_vector[component_id])
        tau_base = anchor_tau[component_id]
        ratio = tau_base / tau_new if tau_new > 0 else 1.0
        # Linear scaling of preventive events with maintenance frequency.
        pm_count = anchor_pm[component_id] * ratio
        # Failure rate inflates with τ: when maintenance is rarer, failures grow.
        cm_count = anchor_cm[component_id] * (tau_new / tau_base) ** float(spec.get('b_M', 1.5))
        # Add an upper bound: at most one corrective per nominal life per printer.
        cm_cap = n_printers * (HORIZON_YEARS * 24 * 365.25 / float(spec['L_nom_h']))
        cm_count = min(cm_count, cm_cap)
        pm_total += pm_count * float(spec['cost_preventive_eur'])
        cm_total += cm_count * float(spec['cost_corrective_eur'])
        downtime_total += pm_count * float(spec['downtime_preventive_h'])
        downtime_total += cm_count * float(spec['downtime_corrective_h'])
    norm = n_printers * HORIZON_YEARS
    annual_cost = (pm_total + cm_total) / norm
    total_hours = len(DATES) * 24.0 * n_printers
    availability = (total_hours - downtime_total) / total_hours if total_hours > 0 else 1.0
    deficit = max(0.0, 0.95 - availability)
    return {
        'value': annual_cost + 1_000_000.0 * deficit,
        'annual_cost': annual_cost,
        'availability': availability,
    }

surrogate_score({c: anchor_tau[c] for c in COMPONENT_IDS})

In [ ]:
TAU_RANGES = {
    'C1': (50.0, 2_000.0),
    'C2': (500.0, 20_000.0),
    'C3': (24.0, 500.0),
    'C4': (100.0, 2_000.0),
    'C5': (500.0, 8_000.0),
    'C6': (1_000.0, 20_000.0),
}

def trial_to_tau(trial: optuna.Trial) -> dict[str, float]:
    return {c: trial.suggest_float(f'tau_{c}', low, high, log=True) for c, (low, high) in TAU_RANGES.items()}

def surrogate_objective(trial: optuna.Trial) -> float:
    tau_vector = trial_to_tau(trial)
    score = surrogate_score(tau_vector)
    for key in ('annual_cost', 'availability'):
        trial.set_user_attr(key, float(score[key]))
    return float(score['value'])

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=99),
)
study.optimize(surrogate_objective, n_trials=500, show_progress_bar=True)
study_df = study.trials_dataframe().sort_values('value').reset_index(drop=True)
study_df.head(5)

In [ ]:
TOP_K = 5
real_results = []
for _, row in study_df.head(TOP_K).iterrows():
    tau_vector = {c: float(row[f'params_tau_{c}']) for c in COMPONENT_IDS}
    events = run_with_tau(
        tau_vector,
        printer_ids=TEST_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    real_score = scalar_objective(events, components_cfg)
    real_results.append({
        'trial': int(row['number']),
        'surrogate_value': float(row['value']),
        'surrogate_annual_cost': float(row['user_attrs_annual_cost']),
        'real_value': float(real_score['value']),
        'real_annual_cost': float(real_score['annual_cost']),
        'real_availability': float(real_score['availability']),
        **{f'tau_{c}': tau_vector[c] for c in COMPONENT_IDS},
    })
real_df = pd.DataFrame(real_results).sort_values('real_value').reset_index(drop=True)
real_df

In [ ]:
winner = real_df.iloc[0]
best_tau = {c: float(winner[f'tau_{c}']) for c in COMPONENT_IDS}
payload = {
    'tau_nom_h': best_tau,
    'validated_on': 'test printers (id 85..99)',
    'surrogate_annual_cost': float(winner['surrogate_annual_cost']),
    'real_annual_cost_eur_per_printer_year': float(winner['real_annual_cost']),
    'real_availability': float(winner['real_availability']),
}
with (RESULTS_DIR / 'best_tau_surrogate.yaml').open('w', encoding='utf-8') as handle:
    yaml.safe_dump(payload, handle, sort_keys=False)
print(yaml.safe_dump(payload, sort_keys=False))